In [1]:
def fm(l):
    #print(l)
    lights=[list(l[0][1:-1])]
    buttons=[[tuple(int(y) for y in x[1:-1].split(',')) for x in l[1:-1]]] 
    counters=[[int(x) for x in l[-1][1:-1].split(',')]]
    return lights+buttons+counters

def load(fn):
    l = open(fn).readlines()
    l=[x.strip().split() for x in l]
    l=list(map(fm,l))
    return l


def v1(x):
    def push(l,B):
        l = l.copy()
        for b in B:
            l[b]='.' if l[b] == '#' else '#'
        return l
    L,B,C=x
    l=["."]*len(L)
    S=[(l,[])]
    past = {}
    while len(S):
        #print("---")
        
        l,h=S[0]
        S=S[1:]
        l_ = "".join(l)
        past["".join(l)]=1
        if "".join(L)=="".join(l):
            return len(h)
        for b in B:
            
            l2 = push(l,b)
            l2_ = "".join(l2)
            if not l2_ in past:
                S.append((l2,h+[b]))
            past[l2_]=1
    return None


def solve(fn, xxx):
    d=load(fn)
    #return xxx(d[0])
    return sum(map(xxx,d))
    
print("part1:",solve("10.txt", v1),479)



part1: 479 479


In [2]:
def v2(x):
    def push(c,B):
        c = c.copy()
        for b in B:
            c[b]+=1
        return c
    
    L,B,C=x
    
    
    c=[0]*len(C)
    S=[(c,[])]
    past = {}
    zzz = 0
    while len(S):
        
        zzz+=1
        if zzz % 1_000_000 == 0:
            print(len(S))
        c,h=S[0]
        S=S[1:]
        
        #print(c, C)
        
        past[tuple(c)]=1
        if C==c:
            return len(h)
        for b in B:
            c2 = push(c,b)
            #print(b, c, c2)
            ok=1
            for i in range(len(c2)):
                if c2[i] > C[i]:
                    ok=0
                    break
            
            if ok and not tuple(c2) in past:
                S.append((c2,h+[b]))
            past[tuple(c2)]=1
    return None

#print("part2:",solve("10.txt", v2),0)




In [3]:
import numpy as np
from scipy.linalg import lu
from sympy import Matrix

def v3(_):
    print(_)
    _,B,C=_
    
    b=np.array([[x] for x in C])
    print("b:\n",b)
    
    l = len(b)
    
    A=[]
    for i in B:
        aa = [0] * l
        for j in i:
            aa[j] = 1
        A.append(aa)
    A=np.array(A).T
    print("A:\n", A)
    
    A_sym = Matrix(A)
    b_sym = Matrix(b)
    
    #x = A_sym.solve_least_squares(b_sym)
    x = A_sym.gauss_jordan_solve(b_sym) 
    print(x)
    return sum(x)


from itertools import product

def ordered_combinations(max_n, length):
    """Yields combinations ordered by maximum value used."""
    # max_val = 0: only (0,0,0,...)
    yield (0,) * length
    
    # max_val = 1, 2, 3, ... up to max_n
    for max_val in range(1, max_n + 1):
        # All combos using digits 0 to max_val, but must include at least one max_val
        for combo in product(range(max_val + 1), repeat=length):
            if max(combo) == max_val:
                yield combo








In [4]:
from z3 import Optimize, Int, Sum, sat

def find_vector_combination(vectors, target):
    """
    Find positive integer coefficients x such that sum(x[i] * vectors[i]) = target,
    minimizing sum(x).
    
    Returns list of coefficients, or None if no solution.
    """
    opt = Optimize()
    x = [Int(f'x_{i}') for i in range(len(vectors))]
    
    for xi in x:
        opt.add(xi >= 0)
    
    for d in range(len(target)):
        opt.add(Sum([x[i] * vectors[i][d] for i in range(len(vectors))]) == target[d])
    
    opt.minimize(Sum(x))
    
    if opt.check() == sat:
        model = opt.model()
        return [model[xi].as_long() for xi in x]
    return None


def v3(x):
    #print(x)
    _,B,C=x
    l = len(C)
    A=[]
    for i in B:
        aa = [0] * l
        for j in i:
            aa[j] = 1
        A.append(aa)
    
    #print("A:\n", A)
    x=find_vector_combination(A, C)
    #print(x)
    return sum(x)

print("part2:",solve("10.txt", v3),19574)





part2: 19574 19574
